# Ćw 13_1 RL
**Reinforcement Learning**
* Kod pochodzi od: [Jeff Heaton](https://sites.wustl.edu/jeffheaton/), McKelvey School of Engineering, [Washington University in St. Louis](https://engineering.wustl.edu/Programs/Pages/default.aspx)


#Do ćwiczeń są filmy video

* Część 12.1: Wprowadzenie do uczenia ze wzmocnieniem [[Video]](https://www.youtube.com/watch?v=FvuyrpzvwdI&list=PLjy4p-07OYzuy_lHcRW8lPTLPTTOmUpmi)
* Część 12.2: Wprowadzenie do Q-Learning [[Video]](https://www.youtube.com/watch?v=VKuqvbG_KAw&list=PLjy4p-07OYzuy_lHcRW8lPTLPTTOmUpmi)
* Część  12.3: Stable Baselines Q-Learning [[Video]](https://www.youtube.com/watch?v=kl7zsCjULN0&list=PLjy4p-07OYzuy_lHcRW8lPTLPTTOmUpmi)
* Część  12.4: Atari Games z Stable Baselines Neural Networks [[Video]](https://www.youtube.com/watch?v=maLA1_d4pzQ&list=PLjy4p-07OYzuy_lHcRW8lPTLPTTOmUpmi)
* Częsć 12.5: Przyszłość/perspektywy Reinforcement Learning [[Video]](https://www.youtube.com/watch?v=-euo5pTjP8E&list=PLjy4p-07OYzuy_lHcRW8lPTLPTTOmUpmi)

# Cześć 12.1: Wprowadzenie do Gymnasium

[Gymnasium](https://github.com/Farama-Foundation/Gymnasium)
ma na celu zapewnienie łatwego w konfiguracji benchmarku inteligencji ogólnej z różnymi środowiskami. Celem jest ujednolicenie sposobu definiowania środowisk w publikacjach dotyczących badań nad sztuczną inteligencją, aby opublikowane badania były łatwiejsze do odtworzenia. Projekt zapewnia użytkownikowi prosty interfejs. Gymnasium jest forkiem OpenAI Gym, dla którego OpenAI zaprzestało wsparcia w październiku 2021 roku. Gymnasium jest obecnie wspierane przez [The Farama Foundation].(https://farama.org/).


Gymnasium jest instalowany przez pip na lokalnym komputerze. Istnieje kilka istotnych ograniczeń, których należy być świadomym:

* Gymnasium Atari tylko **bezpośrednio** wspiera Linux i Macintosh
* Gymnasium Atari może być używany z systemem Windows;
wymaga jednak szczególnej [procedury instalacji](https://towardsdatascience.com/how-to-install-openai-gym-in-a-windows-environment-338969e24d30)

*Gymnasium nie może bezpośrednio renderować animowanych gier w Google CoLab.

Ponieważ Gymnasium wymaga wyświetlania grafiki, osadzone wideo jest jedynym sposobem na wyświetlenie Gymnasium w Google CoLab. Prezentacja animacji gier Gymnasium w Google CoLab została omówiona w dalszej części tego modułu.
## Przegląd środowisk Gymnasium


Centralnym elementem Gymnasium jest środowisko, które definiuje "grę", w której algorytm wzmacniający będzie konkurował. Środowisko nie musi być grą; jednak opisuje następujące cechy podobne do gry:
**przestrzeń akcji**: Jakie działania możemy podjąć w środowisku w każdym kroku/epizodzie, aby zmienić środowisko.
**Przestrzeń obserwacji**: Jaki jest aktualny stan części środowiska, którą możemy obserwować. Zazwyczaj widzimy całe środowisko.

Zanim zaczniemy przyglądać się Gymnasium, konieczne jest zrozumienie terminologii używanej przez tę bibliotekę.


* **Agent** - program lub model uczenia maszynowego, który kontroluje działania.
Krok - jedna runda wykonywania działań, które wpływają na przestrzeń obserwacji.
* **Epizod** - zbiór kroków, który kończy się, gdy agentowi nie uda się osiągnąć celu środowiskowego lub gdy epizod osiągnie maksymalną liczbę dozwolonych kroków.
* **Render** - Gymnasium może wyrenderować jedną klatkę do wyświetlenia po każdym epizodzie.
* **Nagroda** - Pozytywne wzmocnienie, które może pojawić się na końcu każdego epizodu, po działaniu agenta.
* **Niedeterministyczne** - w niektórych środowiskach losowość jest czynnikiem decydującym o wpływie działań na nagrodę i zmiany w przestrzeni obserwacji.

Gymnasium należy zainstalować za pomocą następującego polecenia.


In [ ]:
!pip install gymnasium[accept-rom-license,atari]


Należy zauważyć, że wiele środowisk Gymnasium określa, że nie są one niedeterministyczne, mimo że używają liczb losowych do przetwarzania akcji. W oparciu o Gymnasium GitHub issue tracker, niedeterministyczna właściwość oznacza, że deterministyczne środowisko zachowuje się losowo. Nawet jeśli nadasz środowisku spójną wartość seed, zachowanie to zostanie potwierdzone. Program może użyć metody seed środowiska, aby zasiać generator liczb losowych dla środowiska.

Biblioteka Gymnasium pozwala nam odpytywać niektóre z tych atrybutów ze środowisk. Stworzyłem następującą funkcję do odpytywania środowisk Gymnasium.

In [ ]:
import gymnasium as gym
import ale_py

def query_environment(name):
    env = gym.make(name)
    spec = gym.spec(name)
    print(f"Action Space: {env.action_space}")
    print(f"Observation Space: {env.observation_space}")
    print(f"Max Episode Steps: {spec.max_episode_steps}")
    print(f"Nondeterministic: {spec.nondeterministic}")
    #print(f"Reward Range: {env.reward_range}")
    print(f"Reward Threshold: {spec.reward_threshold}")



Przyjrzymy się środowisku **MountainCar-v0**, które rzuca wyzwanie słabemu samochodowi, aby uciec z doliny między dwiema górami.  Poniższy kod opisuje środowisko Mountian Car.

In [ ]:
query_environment("MountainCar-v0")

Action Space: Discrete(3)
Observation Space: Box([-1.2  -0.07], [0.6  0.07], (2,), float32)
Max Episode Steps: 200
Nondeterministic: False
Reward Threshold: -110.0



Środowisko to pozwala na trzy różne działania: przyspieszanie do przodu, zwalnianie lub cofanie. Przestrzeń obserwacji zawiera dwie wartości ciągłe (zmiennoprzecinkowe), co jest widoczne w obiekcie box. Przestrzeń obserwacji to po prostu pozycja i prędkość samochodu. Samochód ma 200 kroków do ucieczki w każdym odcinku. Musiałbyś spojrzeć na kod, ale samochód górski nie otrzymuje żadnej przyrostowej nagrody. Jedyną nagrodą dla pojazdu jest ucieczka z doliny.

In [ ]:
query_environment("CartPole-v1")

Action Space: Discrete(2)
Observation Space: Box([-4.8               -inf -0.41887903        -inf], [4.8               inf 0.41887903        inf], (4,), float32)
Max Episode Steps: 500
Nondeterministic: False
Reward Threshold: 475.0


Środowisko **CartPole-v1** (Odwrócone wahadło) stawia przed agentem wyzwanie polegające na utrzymaniu równowagi na tyczce. Środowisko ma przestrzeń obserwacji składającą się z 4 ciągłych liczb:

* Pozycja wózka
* Prędkość wózka
* Kąt tyczki
* Prędkość tyczki przy końcówce

Aby osiągnąć ten cel, agent może podjąć następujące działania:

* Pchnięcie wózka w lewo
* Pchnięcie wózka w prawo



Istnieje również ciągły wariant samochodu górskiego. W tej wersji silnik nie jest po prostu włączony lub wyłączony. Przestrzeń akcji jest pojedynczą liczbą zmiennoprzecinkową dla wózka ciągłego, która określa, ile siły do przodu lub do tyłu wózek aktualnie wykorzystuje.

In [ ]:
query_environment("MountainCarContinuous-v0")

Action Space: Box(-1.0, 1.0, (1,), float32)
Observation Space: Box([-1.2  -0.07], [0.6  0.07], (2,), float32)
Max Episode Steps: 999
Nondeterministic: False
Reward Threshold: 90.0



Gymnasium zapewnia wszechstronną platformę do opracowywania i porównywania algorytmów uczenia ze wzmocnieniem. Obsługuje szeroką gamę środowisk, w tym klasyczne gry Atari za pośrednictwem emulatora Arcade Learning Environment (ALE). Integracja ta umożliwia badaczom i entuzjastom dostęp do zestawu gier wideo retro zaprojektowanych pierwotnie dla konsoli Atari 2600, wykorzystując je jako punkty odniesienia dla wydajności sztucznej inteligencji. Łącząc się z ALE, użytkownicy Gymnasium mogą łatwo implementować swoje algorytmy i testować je pod kątem niuansowych wyzwań stawianych przez te gry. Każda gra przedstawia unikalne scenariusze, które mogą pomóc w szkoleniu algorytmów do nauki różnych zadań, dzięki czemu Gymnasium jest nieocenionym narzędziem do rozwijania dziedziny sztucznej inteligencji poprzez te interaktywne i złożone środowiska.

Algorytmy uczenia ze wzmocnieniem (RL) mogą otrzymywać dane wejściowe z gry Atari na dwa podstawowe sposoby, które zaspokajają różne aspekty stanu i złożoności gry.


Pierwsza metoda polega na monitorowaniu „ekranu” gry lub wyjścia wizualnego generowanego przez grę. W tym podejściu algorytm RL przetwarza piksele wyświetlacza gry jako stan środowiska. Jest to podobne do tego, jak ludzki gracz widziałby i interpretował grę. Algorytm analizuje wzorce, ruchy i zmiany w ramkach, aby podejmować decyzje dotyczące najlepszych działań na każdym kroku. Metoda ta wymaga, aby model RL obsługiwał dane wielowymiarowe i nauczył się kojarzyć wskazówki wizualne z wynikami gry.

Druga metoda polega na monitorowaniu pamięci RAM systemu Atari. Pomimo ograniczonej pojemności, pamięć RAM systemu Atari zawiera wszystkie informacje o wewnętrznym stanie gry, takie jak lokalizacja obiektów, wyniki graczy i stan gry. Korzystając bezpośrednio z tej pamięci, algorytm RL może uzyskać dostęp do bardziej zwartej i mniej zaszumionej reprezentacji stanu gry niż dane pikselowe. Może to być korzystne dla bardziej efektywnego uczenia się, ponieważ stan systemu jest reprezentowany w bardziej uporządkowanej i mniej wymiarowej formie.

Obie metody mają swoje zalety. Podejście polegające na przechwytywaniu ekranu zmusza algorytm do uczenia się bezpośrednio z danych wizualnych, co jest podejściem bardziej ogólnym i bliższym temu, jak ludzie grają w gry. Z drugiej strony, metoda monitorowania RAM może prowadzić do szybszego czasu szkolenia i potencjalnie głębszego zrozumienia mechaniki gry, ponieważ pomija potrzebę interpretacji danych wizualnych. Wybór pomiędzy tymi metodami zależy od


Najpierw zobaczymy, jak monitorować ekran gry [Breakout](https://gymnasium.farama.org/environments/atari/breakout/).

In [ ]:
query_environment("ALE/Breakout-v5")

Action Space: Discrete(4)
Observation Space: Box(0, 255, (210, 160, 3), uint8)
Max Episode Steps: None
Nondeterministic: False
Reward Threshold: None



Podobnie możemy monitorować pamięć RAM Breakout.

In [ ]:
#query_environment("ALE/Breakout-ram-v5")


## Renderowanie środowisk Gymnasium OpenAI z CoLab

Możliwe jest wizualizowanie gry, w którą gra agent, nawet w CoLab. Ta sekcja zawiera informacje na temat generowania wideo w CoLab, które pokazuje odcinek gry, w którą gra agent. Oparłem ten proces wideo na sugestiach znalezionych [TU](https://colab.research.google.com/drive/1flu31ulJlgiRL1dnN2ir8wGh9p7Zij2t).

Rozpocznija od **pyvirtualdisplay** i **python-opengl**.

In [ ]:
# HIDE OUTPUT
!pip install pyvirtualdisplay
!sudo apt-get install -y xvfb ffmpeg

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
ffmpeg is already the newest version (7:4.4.2-0ubuntu0.22.04.1).
xvfb is already the newest version (2:21.1.4-2ubuntu1.7~22.04.14).
0 upgraded, 0 newly installed, 0 to remove and 34 not upgraded.


Następnie instalujemy wymagania niezbędne do wyświetlenia gry Atari.

In [ ]:
# HIDE OUTPUT
!sudo apt-get install xvfb

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
xvfb is already the newest version (2:21.1.4-2ubuntu1.7~22.04.14).



Uwaga, powyższa komórka może zażądać ponownego uruchomienia środowiska uruchomieniowego, jeśli tak się stanie, należy ponownie uruchomić środowisko uruchomieniowe CoLab. Następnie definiujemy funkcje używane do wyświetlania wideo, dodając je do notatnika CoLab.


Teraz jesteśmy gotowi do gry.  Używamy prostego losowego agenta.

In [ ]:
import gym
import gymnasium as gymnasium
from gymnasium.wrappers import RecordVideo
import glob
import io
import base64
from IPython.display import HTML
from IPython import display as ipythondisplay
from pyvirtualdisplay import Display

# Start virtual display
display = Display(visible=0, size=(1400, 900))
display.start()

# Create Atlantis environment
env = gymnasium.make('Atlantis-v4', render_mode="rgb_array")
env.metadata['render_fps'] = 30

# Setup the wrapper to record the video

video_callable=lambda episode_id: True
env = RecordVideo(env, video_folder='./videos', episode_trigger=video_callable)

# Reset the environment
env.reset()


# Run the environment until done
terminated = False
truncated = False
while not (terminated or truncated):
    action = env.action_space.sample()  # replace with your own policy!
    obs, reward, terminated, truncated, info = env.step(action)

env.stop_recording()

env.close()

# Display the video
video = io.open(glob.glob('videos/*.mp4')[0], 'r+b').read()
encoded = base64.b64encode(video)
ipythondisplay.display(HTML(data='''
    <video width="640" height="480" controls>
        <source src="data:video/mp4;base64,{0}" type="video/mp4" />
    </video>
'''.format(encoded.decode('ascii'))))



Należy zauważyć, że funkcje **step** i **reset** zwracają kilka wartości:

* **observation** (ObsType): Element przestrzeni obserwacji środowiska observation_space jako następna obserwacja wynikająca z działań agenta. Przykładem jest tablica numpy zawierająca pozycje i prędkości bieguna w CartPole.

* **reward** (SupportsFloat): Nagroda w wyniku podjęcia akcji.

* **terminated** (bool): Czy agent osiągnie stan końcowy (zdefiniowany w MDP zadania), który może być pozytywny lub negatywny. Przykładem jest osiągnięcie stanu docelowego lub przejście do lawy z Sutton i Barton, Gridworld. Jeśli to prawda, użytkownik musi wywołać funkcję reset().

* **truncated** (bool): Czy warunek obcięcia poza zakresem MDP jest spełniony. Zazwyczaj jest to limit czasowy, ale może być również użyty do wskazania fizycznego wyjścia agenta poza granice. Może być użyty do przedwczesnego zakończenia epizodu przed osiągnięciem stanu końcowego. Jeśli true, użytkownik musi wywołać reset().

* **info** (dict): Zawiera pomocnicze informacje diagnostyczne (pomocne przy debugowaniu, uczeniu się i logowaniu). Może to na przykład zawierać: metryki opisujące stan wydajności agenta, zmienne ukryte przed obserwacjami lub indywidualne warunki nagrody, które są łączone w celu uzyskania całkowitej nagrody.